# PANDA - Fixed weights submission notebook

**Settings:** GPU T4 ON, Internet OFF

**Attach on Kaggle:**
1. `prostate-cancer-grade-assessment` (official PANDA competition dataset)
2. B0 ordinal 5-fold weights dataset, containing `efficientnetb0_ordinal_fold{0..4}.pth`
3. B1 ordinal 5-fold weights dataset, containing `efficientnetb1_ordinal_fold{0..4}.pth`
4. `panda-src` or a weights dataset that includes `repo/src`
5. `efficientnetpytorch063` by optimo, or any attached offline dataset containing `efficientnet_pytorch`


In [ ]:
KAGGLE_INPUT = '/kaggle/input'
BATCH_SIZE = 16
NUM_WORKERS = 2
ORDINAL_MODE = 'threshold'
N_FOLDS = 5
OUTPUT_CSV = '/kaggle/working/submission.csv'

# Set True when debugging score quality: no visible matching test images means no real model inference.
# Leave False only when you prefer a valid fallback CSV over a failed Kaggle commit.
REQUIRE_MATCHING_TEST_IMAGES = False

# OOF-backed family list. Missing optional datasets are skipped automatically.
MODEL_FAMILIES = [
    {
        'name': 'b0_ordinal',
        'weights_slugs': [
            '5fold-baseline-ordinal',
            'panda-effnetb0-ordinal-5fold',
            'baseline-thumbnails-b0-ordinal-5fold',
        ],
        'weights_dir': None,
        'backbone': 'efficientnet-b0',
        'weight_pattern': 'efficientnetb0_ordinal_fold{fold}.pth',
        'model_kind': 'baseline',
        'oof_qwk': 0.7244,
    },
    {
        'name': 'b1_ordinal',
        'weights_slugs': [
            'b1-ordinal-5fold',
            'baseline-thumbnails-b1-ordinal-5fold',
        ],
        'weights_dir': None,
        'backbone': 'efficientnet-b1',
        'weight_pattern': 'efficientnetb1_ordinal_fold{fold}.pth',
        'model_kind': 'baseline',
        'oof_qwk': 0.7334,
    },
    {
        'name': 'b0_smoothl1',
        'weights_slugs': [
            'panda-effnetb0-5fold-baseline',
            'baseline-thumbnails-b0-5fold-clean',
            'panda-effnetb0-weights-scaffold',
        ],
        'weights_dir': None,
        'backbone': 'efficientnet-b0',
        'weight_pattern': 'efficientnetb0_fold{fold}.pth',
        'model_kind': 'baseline',
        'oof_qwk': 0.7147,
    },
]

OOF_RECIPES = [
    {
        'name': 'b0+b1+smoothl1_calibrated',
        'requires': ['b0_ordinal', 'b1_ordinal', 'b0_smoothl1'],
        'weights': {'b0_ordinal': 0.20, 'b1_ordinal': 0.54, 'b0_smoothl1': 0.26},
        'thresholds': [0.70, 1.30, 2.25, 3.1167, 4.00],
        'oof_qwk': 0.7508,
    },
    {
        'name': 'b1+smoothl1_calibrated',
        'requires': ['b1_ordinal', 'b0_smoothl1'],
        'weights': {'b1_ordinal': 0.58, 'b0_smoothl1': 0.42},
        'thresholds': [0.75, 1.21, 2.20, 3.15, 4.00],
        'oof_qwk': 0.7480,
    },
    {
        'name': 'b0+b1_calibrated',
        'requires': ['b0_ordinal', 'b1_ordinal'],
        'weights': {'b0_ordinal': 0.395, 'b1_ordinal': 0.605},
        'thresholds': [0.65, 1.25, 2.20, 3.00, 4.25],
        'oof_qwk': 0.7451,
    },
    {
        'name': 'b0+smoothl1_calibrated',
        'requires': ['b0_ordinal', 'b0_smoothl1'],
        'weights': {'b0_ordinal': 0.42, 'b0_smoothl1': 0.58},
        'thresholds': [0.40, 1.35, 2.35, 3.20, 3.85],
        'oof_qwk': 0.7363,
    },
    {
        'name': 'b1_ordinal_single',
        'requires': ['b1_ordinal'],
        'weights': {'b1_ordinal': 1.0},
        'thresholds': [0.50, 1.50, 2.50, 3.50, 4.50],
        'oof_qwk': 0.7334,
    },
    {
        'name': 'b0_smoothl1_calibrated',
        'requires': ['b0_smoothl1'],
        'weights': {'b0_smoothl1': 1.0},
        'thresholds': [0.5333, 1.50, 2.2667, 3.0167, 4.05],
        'oof_qwk': 0.7289,
    },
    {
        'name': 'b0_ordinal_single',
        'requires': ['b0_ordinal'],
        'weights': {'b0_ordinal': 1.0},
        'thresholds': [0.50, 1.50, 2.50, 3.50, 4.50],
        'oof_qwk': 0.7244,
    },
]


In [ ]:
import glob
import importlib.util
import os
import sys
import traceback

SETUP_ERRORS = []
SRC_READY = False
MODEL_READY = False

def record_warning(label, exc=None):
    message = f'WARN: {label}'
    if exc is not None:
        message += f': {exc}'
    print(message)
    SETUP_ERRORS.append(message)

def find_existing_path(patterns, description, required=True):
    matches = []
    for pattern in patterns:
        if pattern is None:
            continue
        try:
            if any(ch in pattern for ch in '*?['):
                matches.extend(glob.glob(pattern, recursive=True))
            elif os.path.exists(pattern):
                matches.append(pattern)
        except Exception as exc:
            record_warning(f'Could not evaluate path pattern for {description}: {pattern}', exc)
    matches = sorted(set(p for p in matches if os.path.exists(p)), key=lambda p: (len(p), p))
    if matches:
        return matches[0]
    checked = '\n  - '.join(str(p) for p in patterns)
    message = f'Could not find {description}. Checked:\n  - {checked}'
    if required:
        record_warning(message)
    return None

def add_efficientnet_paths():
    added = []

    package_inits = glob.glob(os.path.join(KAGGLE_INPUT, '**', 'efficientnet_pytorch', '__init__.py'), recursive=True)
    for init_path in sorted(package_inits, key=lambda p: (len(p), p)):
        package_parent = os.path.dirname(os.path.dirname(init_path))
        if package_parent not in sys.path:
            sys.path.insert(0, package_parent)
            added.append(package_parent)

    wheel_paths = glob.glob(os.path.join(KAGGLE_INPUT, '**', '*efficientnet*.whl'), recursive=True)
    for wheel_path in sorted(wheel_paths, key=lambda p: (len(p), p)):
        if wheel_path not in sys.path:
            sys.path.insert(0, wheel_path)
            added.append(wheel_path)

    return added

try:
    efficientnet_paths = add_efficientnet_paths()
    efficientnet_spec = importlib.util.find_spec('efficientnet_pytorch')
    print('EfficientNet search paths:', efficientnet_paths or 'none found in attached inputs')
    print('EfficientNet import location:', efficientnet_spec.origin if efficientnet_spec else 'not importable yet')
except Exception as exc:
    efficientnet_paths = []
    efficientnet_spec = None
    record_warning('EfficientNet path discovery failed', exc)


In [ ]:
import shutil

np = pd = torch = cv2 = skimage = None
round_preds = None
load_model = None
predict = None
device = None

try:
    import numpy as np
    import pandas as pd
    import torch
    import cv2
    import skimage.io
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Device:', device)
except Exception as exc:
    record_warning('Core Python package import failed; inference will be skipped and sample_submission.csv will be written', exc)

def find_src_dir():
    patterns = [
        os.path.join(KAGGLE_INPUT, 'panda-src', 'src', 'inference.py'),
        os.path.join(KAGGLE_INPUT, 'panda-src', 'inference.py'),
        os.path.join(KAGGLE_INPUT, '**', 'repo', 'src', 'inference.py'),
        os.path.join(KAGGLE_INPUT, '**', 'src', 'inference.py'),
        os.path.join(KAGGLE_INPUT, '**', 'inference.py'),
    ]
    inference_files = []
    for pattern in patterns:
        try:
            if any(ch in pattern for ch in '*?['):
                inference_files.extend(glob.glob(pattern, recursive=True))
            elif os.path.exists(pattern):
                inference_files.append(pattern)
        except Exception as exc:
            record_warning(f'Could not search for src pattern {pattern}', exc)
    inference_files = sorted(set(inference_files), key=lambda p: (0 if '/repo/src/' in p else 1, len(p), p))

    expected = ['eval.py', 'model.py', 'dataset.py']
    checked = []
    for inference_file in inference_files:
        src_dir = os.path.dirname(inference_file)
        missing = [name for name in expected if not os.path.exists(os.path.join(src_dir, name))]
        checked.append((src_dir, missing))
        if not missing:
            print('Using src package:', src_dir)
            return src_dir

    details = '\n'.join(f'  - {src_dir}: missing {missing}' for src_dir, missing in checked)
    if not details:
        details = '  - no candidate inference.py files found'
    record_warning(
        'Could not find a complete panda src package with inference.py, eval.py, model.py, and dataset.py.\n'
        f'Checked:\n{details}'
    )
    return None

try:
    working_src = '/kaggle/working/src'
    src_dir = find_src_dir()
    if src_dir is not None:
        if os.path.exists(working_src):
            shutil.rmtree(working_src)
        shutil.copytree(src_dir, working_src)
        sys.path.insert(0, '/kaggle/working')
        SRC_READY = True
        print('Copied src from:', src_dir)
    else:
        SRC_READY = False
except Exception as exc:
    SRC_READY = False
    record_warning('Could not copy src package into /kaggle/working/src', exc)

def has_all_fold_weights(weights_dir, family):
    missing = []
    for fold in range(N_FOLDS):
        weight_name = family['weight_pattern'].format(fold=fold)
        if not os.path.exists(os.path.join(weights_dir, weight_name)):
            missing.append(weight_name)
    if missing:
        print(f"WARN: {family['name']} weights in {weights_dir} are incomplete; missing {missing}")
        return False
    return True

def family_weight_slugs(family):
    slugs = family.get('weights_slugs')
    if slugs is None:
        slugs = [family.get('weights_slug')]
    return [slug for slug in slugs if slug]


def find_weights_dir(family):
    slug_patterns = []
    for slug in family_weight_slugs(family):
        slug_patterns.extend([
            os.path.join(KAGGLE_INPUT, slug),
            os.path.join(KAGGLE_INPUT, '**', slug),
        ])
    by_slug = find_existing_path(
        slug_patterns,
        f"{family['name']} weights dataset",
        required=False,
    )
    if by_slug is not None and has_all_fold_weights(by_slug, family):
        return by_slug

    first_weight = family['weight_pattern'].format(fold=0)
    weight_paths = glob.glob(os.path.join(KAGGLE_INPUT, '**', first_weight), recursive=True)
    for weight_path in sorted(weight_paths, key=lambda p: (len(p), p)):
        weights_dir = os.path.dirname(weight_path)
        if has_all_fold_weights(weights_dir, family):
            return weights_dir
    print(f"WARN: {family['name']} complete 5-fold weights were not found; this family will be skipped")
    return None

try:
    for family in MODEL_FAMILIES:
        family['weights_dir'] = find_weights_dir(family)
        print(f"{family['name']} weights:", family['weights_dir'])
except Exception as exc:
    record_warning('Weight discovery failed; inference will be skipped if no complete family is usable', exc)

try:
    if SRC_READY:
        import efficientnet_pytorch
        from src.eval import round_preds
        from src.inference import load_model, predict
        MODEL_READY = True
        print('Model code import: ready')
    else:
        MODEL_READY = False
        record_warning('Model code import skipped because src package is not ready')
except Exception as exc:
    MODEL_READY = False
    record_warning('Model code import failed; sample_submission.csv will be written instead of raising', exc)

if round_preds is None and np is not None:
    def round_preds(preds, num_classes=6, ordinal_mode='threshold'):
        return np.clip(np.round(np.asarray(preds)), 0, num_classes - 1).astype(int)


In [ ]:
# Dataset that reads resized PNGs or official TIFF slide images directly
if np is None or torch is None or cv2 is None or skimage is None:
    record_warning('Image dataset helpers skipped because one or more core packages are unavailable')
else:
    IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    IMAGE_EXTS = ('.png', '.tiff', '.tif')
    PANDA_IMAGE_DIR_CANDIDATES = [
        os.path.join(KAGGLE_INPUT, 'prostate-cancer-grade-assessment', 'test_images'),
        os.path.join('..', 'input', 'prostate-cancer-grade-assessment', 'test_images'),
        os.path.join(KAGGLE_INPUT, 'competitions', 'prostate-cancer-grade-assessment', 'test_images'),
        os.path.join('..', 'input', 'competitions', 'prostate-cancer-grade-assessment', 'test_images'),
        os.path.join(KAGGLE_INPUT, 'prostate-cancer-grade-assessment', 'train_images'),
        os.path.join('..', 'input', 'prostate-cancer-grade-assessment', 'train_images'),
        os.path.join(KAGGLE_INPUT, 'competitions', 'prostate-cancer-grade-assessment', 'train_images'),
        os.path.join('..', 'input', 'competitions', 'prostate-cancer-grade-assessment', 'train_images'),
    ]

    def slide_path(image_dir, image_id):
        for ext in IMAGE_EXTS:
            path = os.path.join(image_dir, f'{image_id}{ext}')
            if os.path.exists(path):
                return path
        return None

    def coerce_rgb_image(img):
        img = np.asarray(img)
        if img.ndim == 2:
            img = np.repeat(img[..., None], 3, axis=2)
        if img.shape[-1] == 4:
            img = img[..., :3]
        if img.shape[-1] != 3:
            raise ValueError(f'Expected an RGB image, got shape {tuple(img.shape)}')
        return np.ascontiguousarray(img)

    def normalize_image(img):
        img = coerce_rgb_image(img)
        if np.issubdtype(img.dtype, np.integer):
            img = img.astype(np.float32) / np.iinfo(img.dtype).max
        else:
            img = img.astype(np.float32)
            if img.max() > 1.0:
                img = img / 255.0
        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        return np.ascontiguousarray(img.transpose(2, 0, 1))

    def read_slide_image(path):
        ext = os.path.splitext(path)[1].lower()
        if ext in ('.tiff', '.tif'):
            return skimage.io.MultiImage(path)[-1]
        return skimage.io.imread(path)

    def count_matching_images(image_dir, image_ids):
        if image_dir is None or not os.path.isdir(image_dir):
            return 0
        return sum(slide_path(image_dir, image_id) is not None for image_id in image_ids)

    def candidate_image_dirs(preferred_dir=None):
        candidates = []
        for path in [preferred_dir, *PANDA_IMAGE_DIR_CANDIDATES]:
            if path is not None and os.path.isdir(path):
                candidates.append(path)

        for root in [KAGGLE_INPUT, os.path.join('..', 'input')]:
            for leaf in ['test_images', 'train_images']:
                candidates.extend(
                    path for path in glob.glob(os.path.join(root, '**', leaf), recursive=True)
                    if os.path.isdir(path)
                )
        return sorted(set(candidates), key=lambda p: (len(p), p))

    def find_image_dir_for_ids(image_ids, preferred_dir=None):
        image_ids = [str(image_id) for image_id in image_ids]
        candidates = candidate_image_dirs(preferred_dir)
        if not candidates:
            print('PANDA image dir check: no test_images or train_images directory found')
            return None

        scored = [(count_matching_images(path, image_ids), path) for path in candidates]
        scored = sorted(scored, key=lambda item: (-item[0], len(item[1]), item[1]))
        best_count, best_path = scored[0]
        print(f'PANDA image dir check: {best_path} ({best_count}/{len(image_ids)} matched)')
        if best_count == len(image_ids):
            return best_path
        print('Visible image dirs are absent or incomplete for these IDs; keeping sample_submission.csv')
        return None

    class SlideImageDataset(torch.utils.data.Dataset):
        def __init__(self, df, image_dir):
            self.df = df.reset_index(drop=True)
            self.image_dir = image_dir
        def __len__(self):
            return len(self.df)
        def __getitem__(self, i):
            row = self.df.iloc[i]
            path = slide_path(self.image_dir, row.image_id)
            if path is None:
                raise FileNotFoundError(f'Could not find image for {row.image_id} in {self.image_dir}')
            img = read_slide_image(path)
            img = cv2.resize(img, (512, 512))
            img = normalize_image(img)
            return torch.from_numpy(img), torch.tensor(0.0)


In [ ]:
SUBMISSION_COLUMNS = ['image_id', 'isup_grade']


def make_empty_submission():
    return pd.DataFrame({'image_id': [], 'isup_grade': []})


def format_submission(df):
    df = df.copy()
    if 'image_id' not in df.columns:
        df['image_id'] = ''
    if 'isup_grade' not in df.columns:
        df['isup_grade'] = 0
    df = df[SUBMISSION_COLUMNS]
    df['image_id'] = df['image_id'].astype(str)
    df['isup_grade'] = pd.to_numeric(df['isup_grade'], errors='coerce').fillna(0)
    df['isup_grade'] = df['isup_grade'].round().clip(0, 5).astype(int)
    return df


def load_fallback_submission(sample_path):
    if sample_path is not None and os.path.exists(sample_path):
        return pd.read_csv(sample_path)
    return make_empty_submission()


def provider_prior_predictions(test_df, train_path):
    if train_path is None or not os.path.exists(train_path):
        print('No train.csv available for metadata prior fallback; keeping sample_submission.csv')
        return None
    if 'data_provider' not in test_df.columns:
        print('test.csv has no data_provider column; keeping sample_submission.csv')
        return None

    train_df = pd.read_csv(train_path)
    required = {'data_provider', 'isup_grade'}
    if not required.issubset(train_df.columns):
        print('train.csv is missing data_provider/isup_grade; keeping sample_submission.csv')
        return None

    priors = train_df.groupby('data_provider')['isup_grade'].median().round().clip(0, 5).astype(int).to_dict()
    default_grade = int(round(float(train_df['isup_grade'].mean())))
    default_grade = int(np.clip(default_grade, 0, 5))
    print('Using data_provider prior fallback:', priors, 'default:', default_grade)
    return test_df['data_provider'].map(priors).fillna(default_grade).astype(int).values


def choose_oof_recipe(family_preds):
    available = set(family_preds)
    for recipe in sorted(OOF_RECIPES, key=lambda item: item['oof_qwk'], reverse=True):
        if set(recipe['requires']).issubset(available):
            return recipe
    return None


def apply_thresholds(values, thresholds):
    return np.digitize(np.asarray(values, dtype=np.float32), thresholds).astype(int)


def combine_family_predictions(family_preds):
    recipe = choose_oof_recipe(family_preds)
    if recipe is None:
        raise FileNotFoundError('No OOF recipe can be used with the attached model families')
    combined = np.zeros_like(next(iter(family_preds.values())), dtype=np.float32)
    for name, weight in recipe['weights'].items():
        combined += float(weight) * family_preds[name].astype(np.float32)
    final_preds = apply_thresholds(combined, recipe['thresholds'])
    final_preds = np.clip(final_preds, 0, 5).astype(int)
    print('Selected OOF recipe:', recipe['name'], 'OOF QWK:', recipe['oof_qwk'])
    print('Recipe weights:', recipe['weights'])
    print('Recipe thresholds:', recipe['thresholds'])
    return final_preds


try:
    DATA = find_existing_path(
        globals().get('PANDA_IMAGE_DIR_CANDIDATES', []),
        'PANDA test_images or train_images directory',
        required=False,
    )
    TEST = find_existing_path(
        [
            os.path.join(KAGGLE_INPUT, 'prostate-cancer-grade-assessment', 'test.csv'),
            os.path.join('..', 'input', 'prostate-cancer-grade-assessment', 'test.csv'),
            os.path.join(KAGGLE_INPUT, 'competitions', 'prostate-cancer-grade-assessment', 'test.csv'),
            os.path.join('..', 'input', 'competitions', 'prostate-cancer-grade-assessment', 'test.csv'),
            os.path.join(KAGGLE_INPUT, '**', 'test.csv'),
            os.path.join('..', 'input', '**', 'test.csv'),
        ],
        'PANDA test.csv',
        required=False,
    )
    SAMPLE = find_existing_path(
        [
            os.path.join(KAGGLE_INPUT, 'prostate-cancer-grade-assessment', 'sample_submission.csv'),
            os.path.join('..', 'input', 'prostate-cancer-grade-assessment', 'sample_submission.csv'),
            os.path.join(KAGGLE_INPUT, 'competitions', 'prostate-cancer-grade-assessment', 'sample_submission.csv'),
            os.path.join('..', 'input', 'competitions', 'prostate-cancer-grade-assessment', 'sample_submission.csv'),
            os.path.join(KAGGLE_INPUT, '**', 'sample_submission.csv'),
            os.path.join('..', 'input', '**', 'sample_submission.csv'),
        ],
        'PANDA sample_submission.csv',
        required=False,
    )
    TRAIN = find_existing_path(
        [
            os.path.join(KAGGLE_INPUT, 'prostate-cancer-grade-assessment', 'train.csv'),
            os.path.join('..', 'input', 'prostate-cancer-grade-assessment', 'train.csv'),
            os.path.join(KAGGLE_INPUT, 'competitions', 'prostate-cancer-grade-assessment', 'train.csv'),
            os.path.join('..', 'input', 'competitions', 'prostate-cancer-grade-assessment', 'train.csv'),
            os.path.join(KAGGLE_INPUT, '**', 'train.csv'),
            os.path.join('..', 'input', '**', 'train.csv'),
        ],
        'PANDA train.csv',
        required=False,
    )
    print('DATA:', DATA)
    print('TEST:', TEST)
    print('SAMPLE:', SAMPLE)
    print('TRAIN:', TRAIN)

    sub_df = load_fallback_submission(SAMPLE)

    if TEST is None:
        test_df = sub_df[['image_id']].copy()
    else:
        test_df = pd.read_csv(TEST)
        if 'image_id' not in test_df.columns:
            raise ValueError(f'test.csv missing image_id column: {test_df.columns.tolist()}')
    test_ids = test_df[['image_id']].copy()
    test_ids['isup_grade'] = 0

    if 'find_image_dir_for_ids' in globals():
        DATA = find_image_dir_for_ids(test_ids.image_id.values, DATA)
    else:
        print('Image directory helper unavailable; using metadata/sample fallback')
        DATA = None
    print('RESOLVED_DATA:', DATA)

    if os.path.exists(DATA or ''):
        test_dataset = SlideImageDataset(test_ids, DATA)
        print('Test slides:', len(test_dataset))

        usable_families = [family for family in MODEL_FAMILIES if family.get('weights_dir') is not None]
        print('Usable model families:', [family['name'] for family in usable_families])
        if not usable_families:
            raise FileNotFoundError('No usable model weights were found in attached Kaggle inputs')

        family_preds = {}
        for family in usable_families:
            fold_preds = []
            for fold in range(N_FOLDS):
                weight_path = os.path.join(
                    family['weights_dir'],
                    family['weight_pattern'].format(fold=fold)
                )
                model = load_model(
                    weight_path,
                    backbone=family.get('backbone', 'efficientnet-b0'),
                    device=device,
                    model_kind=family.get('model_kind', 'baseline'),
                )
                preds = predict(
                    model, test_dataset, device,
                    batch_size=family.get('batch_size', BATCH_SIZE),
                    num_workers=family.get('num_workers', NUM_WORKERS),
                    ordinal_mode=ORDINAL_MODE,
                )
                fold_preds.append(preds)
                del model
                if device.type == 'cuda':
                    torch.cuda.empty_cache()
                print(f"{family['name']} fold {fold} done")
            family_preds[family['name']] = np.mean(fold_preds, axis=0).astype(np.float32)
            print(f"{family['name']} family prediction mean:", float(np.mean(family_preds[family['name']])))

        final_preds = combine_family_predictions(family_preds)
        print('REAL_INFERENCE_OCCURRED: True')
        sub_df = pd.DataFrame({'image_id': test_ids.image_id.values, 'isup_grade': final_preds})
    else:
        print('No matching image files found for test.csv rows; using metadata prior fallback')
        print('REAL_INFERENCE_OCCURRED: False')
        print('Score from this run is fallback-only and should not be treated as model performance.')
        if REQUIRE_MATCHING_TEST_IMAGES:
            raise FileNotFoundError(
                'No visible image files match test.csv rows, so model inference cannot run. '
                'Attach a dataset with matching test images or use OOF/CV evaluation instead.'
            )
        prior_preds = provider_prior_predictions(test_df, TRAIN)
        if prior_preds is not None:
            sub_df = pd.DataFrame({'image_id': test_df.image_id.values, 'isup_grade': prior_preds})
        else:
            print('Metadata prior unavailable; keeping sample_submission.csv')
except Exception as exc:
    import traceback
    print('Error:', exc)
    traceback.print_exc()
    sub_df = load_fallback_submission(locals().get('SAMPLE'))

sub_df = format_submission(sub_df)
sub_df.to_csv('submission.csv', index=False)
print('Saved submission.csv')
print('Submission columns:', sub_df.columns.tolist())
print('Submission rows:', len(sub_df))
print(sub_df.head())
